In [1]:
# Backup codes

def process_diagnostic_data(trial_df, round_df, group_df):
    
    group_conditions = group_df[['group_id', 'experimental_condition', 'domain_order']].copy()
    
    # 6. Calculate performance in diagnostic tests for each round
    # Create a copy of the round dataframe
    trial_df_analysis = trial_df.copy()

    trial_df_analysis = trial_df_analysis[trial_df_analysis['interaction_type']=='diagnostic test']

    # Calculate proportion of optimal responses for each group, domain, kc_id, and round
    if 'is_opt_response' in trial_df_analysis.columns:
        # Group by the combination of group_id, domain, kc_id, and round
        diagnostic_performance = (trial_df_analysis
                                .groupby(['user_id','group_id', 'domain', 'kc_id', 'round'])
                                .agg(
                                    optimal_response_count=('is_opt_response', 'sum'),
                                    total_tests=('is_opt_response', 'count'),
                                )
                                .reset_index())
        
        # Calculate the proportion of optimal responses
        diagnostic_performance['optimal_response_ratio'] = (
            diagnostic_performance['optimal_response_count'] / 
            diagnostic_performance['total_tests']
        )
        
        # Fill NaN values that might occur if total_tests is 0
        diagnostic_performance['optimal_response_ratio'] = diagnostic_performance['optimal_response_ratio'].fillna(0)
        

    diagnostic_performance = diagnostic_performance.merge(
        group_conditions,
        on='group_id',
        how='left'
    )
    # print(diagnostic_performance)
    # diagnostic_performance.drop_duplicates(subset=['user_id', 'domain', 'round', 'iteration'])
    diagnostic_performance.to_excel('../results/diagnostic_performance_updated.xlsx', index=False)
    


## Plotting

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load the Excel file
file_path = '../results/expanded_combined_metrics_analysis.xlsx'
df = pd.read_excel(file_path)

# Set a consistent style for all plots
sns.set(style="whitegrid")
plt.rcParams.update({'font.size': 12, 'figure.figsize': (12, 8)})

# 1. Plot histogram of total_rounds for all data
plt.figure()
# Calculate appropriate bin edges for integer data
min_rounds = max(0, int(df['total_rounds'].min()))
max_rounds = int(df['total_rounds'].max()) 
bin_edges = np.arange(min_rounds - 0.5, max_rounds + 1.5, 1)

sns.histplot(data=df, x='total_rounds', kde=True, color='blue', 
             stat='probability', binwidth=1, binrange=(min_rounds-0.5, max_rounds+0.5),
             discrete=True)
plt.title('Distribution of Total Rounds (All Data)')
plt.xlabel('Total Rounds')
plt.ylabel('Probability')
plt.xlim(left=0)  # Set x-axis to start at 0
plt.tight_layout()
plt.savefig('histogram_total_rounds_all.png')
plt.close()

# 2. Plot histogram of total_correct for all data
plt.figure()
# Calculate appropriate bin edges for integer data
min_correct = max(0, int(df['total_correct'].min()))
max_correct = int(df['total_correct'].max())
bin_edges_correct = np.arange(min_correct - 0.5, max_correct + 1.5, 1)

sns.histplot(data=df, x='total_correct', kde=True, color='green', 
             stat='probability', binwidth=1, binrange=(min_correct-0.5, max_correct+0.5),
             discrete=True)
plt.title('Distribution of Total Correct Answers (All Data)')
plt.xlabel('Total Correct Answers')
plt.ylabel('Probability')
plt.xlim(left=0)  # Set x-axis to start at 0
plt.tight_layout()
plt.savefig('histogram_total_correct_all.png')
plt.close()

# 3. Plot histograms of total_rounds for each experimental condition
plt.figure()
# Get unique experimental conditions
conditions = df['experimental_condition'].unique()
# Calculate overall min and max for consistent binning across conditions
min_rounds = max(0, int(df['total_rounds'].min()))
max_rounds = int(df['total_rounds'].max())

for condition in conditions:
    condition_data = df[df['experimental_condition'] == condition]
    sns.histplot(data=condition_data, x='total_rounds', kde=True, 
                 label=f'Condition: {condition}', alpha=0.7, stat='probability',
                 binwidth=1, binrange=(min_rounds-0.5, max_rounds+0.5),
                 discrete=True)
    
plt.title('Distribution of Total Rounds by Experimental Condition')
plt.xlabel('Total Rounds')
plt.ylabel('Probability')
plt.legend()
plt.xlim(left=0)  # Set x-axis to start at 0
    
plt.title('Distribution of Total Rounds by Experimental Condition')
plt.xlabel('Total Rounds')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.savefig('histogram_total_rounds_by_condition.png')
plt.close()

# 4. Plot histograms of total_correct for each experimental condition
plt.figure()
# Calculate overall min and max for consistent binning across conditions
min_correct = max(0, int(df['total_correct'].min()))
max_correct = int(df['total_correct'].max())

for condition in conditions:
    condition_data = df[df['experimental_condition'] == condition]
    sns.histplot(data=condition_data, x='total_correct', kde=True, 
                 label=f'Condition: {condition}', alpha=0.7, stat='probability',
                 binwidth=1, binrange=(min_correct-0.5, max_correct+0.5),
                 discrete=True)
    
plt.title('Distribution of Total Correct Answers by Experimental Condition')
plt.xlabel('Total Correct Answers')
plt.ylabel('Probability')
plt.legend()
plt.xlim(left=0)  # Set x-axis to start at 0
    
plt.title('Distribution of Total Correct Answers by Experimental Condition')
plt.xlabel('Total Correct Answers')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.savefig('histogram_total_correct_by_condition.png')
plt.close()

# 5. Create a grid of histograms for easier comparison (FacetGrid)
# For total_rounds
g = sns.FacetGrid(df, col="experimental_condition", col_wrap=2, height=4, aspect=1.5)
# For total_rounds in facetgrid
min_rounds = max(0, int(df['total_rounds'].min()))
max_rounds = int(df['total_rounds'].max())

g = sns.FacetGrid(df, col="experimental_condition", col_wrap=2, height=4, aspect=1.5)
g.map_dataframe(sns.histplot, x="total_rounds", kde=True, stat="probability",
               binwidth=1, binrange=(min_rounds-0.5, max_rounds+0.5), discrete=True)
g.set_axis_labels("Total Rounds", "Probability")
g.set_titles("{col_name}")
# Set x-axis to start at 0 for all subplots
for ax in g.axes.flat:
    ax.set_xlim(left=0)
g.tight_layout()
g.set_titles("{col_name}")
g.tight_layout()
plt.savefig('facetgrid_total_rounds.png')
plt.close()

# For total_correct
g = sns.FacetGrid(df, col="experimental_condition", col_wrap=2, height=4, aspect=1.5)
# For total_correct in facetgrid
min_correct = max(0, int(df['total_correct'].min())) 
max_correct = int(df['total_correct'].max())

g = sns.FacetGrid(df, col="experimental_condition", col_wrap=2, height=4, aspect=1.5)
g.map_dataframe(sns.histplot, x="total_correct", kde=True, stat="probability",
               binwidth=1, binrange=(min_correct-0.5, max_correct+0.5), discrete=True)
g.set_axis_labels("Total Correct Answers", "Probability")
g.set_titles("{col_name}")
# Set x-axis to start at 0 for all subplots
for ax in g.axes.flat:
    ax.set_xlim(left=0)
g.tight_layout()
g.set_titles("{col_name}")
g.tight_layout()
plt.savefig('facetgrid_total_correct.png')
plt.close()

# 6. Optional: Add descriptive statistics to understand the data better
print("\nDescriptive Statistics for Total Rounds:")
print(df.groupby('experimental_condition')['total_rounds'].describe())

print("\nDescriptive Statistics for Total Correct:")
print(df.groupby('experimental_condition')['total_correct'].describe())

# 7. Optional: Create a violin plot for a different visualization
plt.figure(figsize=(14, 8))
sns.violinplot(x='experimental_condition', y='total_rounds', data=df)
plt.title('Violin Plot of Total Rounds by Experimental Condition')
plt.xlabel('Experimental Condition')
plt.ylabel('Total Rounds')
plt.xticks(rotation=45)
plt.ylim(bottom=0)  # Set y-axis to start at 0
plt.tight_layout()
plt.savefig('violin_total_rounds.png')
plt.close()

plt.figure(figsize=(14, 8))
sns.violinplot(x='experimental_condition', y='total_correct', data=df)
plt.title('Violin Plot of Total Correct Answers by Experimental Condition')
plt.xlabel('Experimental Condition')
plt.ylabel('Total Correct Answers')
plt.xticks(rotation=45)
plt.ylim(bottom=0)  # Set y-axis to start at 0
plt.tight_layout()
plt.savefig('violin_total_correct.png')
plt.close()

print("All plots have been saved as PNG files in the current directory.")

In [ ]:
def parse_trial_mdp_info(domain, trial_str):
    mdp_class = MDP_CLASSES.get(domain)
    if not mdp_class:
        raise ValueError(f"Unknown domain: {domain}")
    trial_mdp_dict = ast.literal_eval(trial_str)
    env_idx = trial_mdp_dict.get('env_traj_idxs', [None])[0]
    opt_moves = trial_mdp_dict.get('opt_actions', [])
    return env_idx, opt_moves, trial_mdp_dict


def calculate_regret_metrics(trial_data, make_custom_mdp, rollout_func):
    domain = trial_data['domain']
    gt_weights = GT_WEIGHTS[domain]

    env_idx, opt_moves, trial_mdp_dict = parse_trial_mdp_info(domain, trial_data['mdp_parameters'])
    human_moves = ast.literal_eval(trial_data['moves'])
    est_weights = normalize_reward_weights(trial_data['reward_ft_weights'])

    trial_mdp_dict['weights'] = gt_weights
    mdp = make_custom_mdp(MDP_CLASSES[domain], trial_mdp_dict)
    cur_state = mdp.get_init_state()

    opt_traj = simulate_trajectory(mdp, cur_state, opt_moves, gt_weights)
    mdp.reset()
    human_traj = simulate_trajectory(mdp, cur_state, human_moves, gt_weights)
    mdp.reset()

    phi_opt = mdp.accumulate_reward_features(opt_traj, discount=True)
    mdp.reset()
    phi_human = mdp.accumulate_reward_features(human_traj, discount=True)
    mdp.reset()

    mdp.weights = est_weights
    est_traj = rollout_func(mdp, cur_state, est_weights)
    mdp.reset()
    phi_est = mdp.accumulate_reward_features(est_traj, discount=True)

    human_reward_gt = float(gt_weights.dot(phi_human.T))
    opt_reward_gt = float(gt_weights.dot(phi_opt.T))
    human_reward_est = float(est_weights.dot(phi_human.T))
    opt_reward_est = float(est_weights.dot(phi_est.T))

    return {
        "opt_reward_gt_mdp": opt_reward_gt,
        "human_reward_gt_mdp": human_reward_gt,
        "opt_reward_est_mdp": opt_reward_est,
        "human_reward_est_mdp": human_reward_est,
        "traj_regret_norm": (opt_reward_gt - human_reward_gt) / (opt_reward_gt + 1e-4),
        "est_regret_norm": (opt_reward_est - human_reward_est) / (opt_reward_est + 1e-4),
        "sign_agreement": np.all(np.sign(gt_weights) == np.sign(est_weights)),
        "phi_opt": phi_opt,
        "phi_human": phi_human,
        "phi_est": phi_est,
        "est_traj": est_traj
    }


In [ ]:
# Step 7: Create visualizations of the combined data
try:
    import os
    os.makedirs('combined_analysis', exist_ok=True)
    
    # 1. Correlation heatmap of all numeric variables
    plt.figure(figsize=(14, 12))
    correlation_data = combined_df.select_dtypes(include=['number']).corr()
    mask = np.triu(np.ones_like(correlation_data, dtype=bool))
    
    sns.heatmap(
        correlation_data,
        mask=mask,
        annot=True,
        cmap='coolwarm',
        vmin=-1,
        vmax=1,
        center=0,
        fmt='.2f'
    )
    plt.title('Correlation Between All Metrics', fontsize=16)
    plt.tight_layout()
    plt.savefig('combined_analysis/all_metrics_correlation.png', dpi=300)
    plt.close()
    
    # 2. Scatter plots of key relationships with regression lines
    if 'attention' in combined_df.columns and 'total_correct' in combined_df.columns:
        plt.figure(figsize=(10, 6))
        sns.scatterplot(
            data=combined_df,
            x='attention',
            y='total_correct',
            hue='domain',
            alpha=0.7
        )
        sns.regplot(
            data=combined_df,
            x='attention',
            y='total_correct',
            scatter=False,
            color='black'
        )
        plt.title('Relationship Between Attention and Test Performance', fontsize=14)
        plt.tight_layout()
        plt.savefig('combined_analysis/attention_vs_performance.png', dpi=300)
        plt.close()
    
    # 3. Grouped bar charts comparing metrics across experimental conditions with standard error bars
    if 'experimental_condition' in combined_df.columns:
        # Create a summary by experimental condition and domain with mean and standard error
        condition_metrics = []
        
        for metric in ['attention', 'usability', 'understanding', 'total_correct', 'total_rounds', 'accuracy_ratio']:
            if metric in combined_df.columns:
                # Calculate mean and standard error for each combination
                summary = combined_df.groupby(['experimental_condition', 'domain'])[metric].agg([
                    ('mean', 'mean'),
                    ('sem', lambda x: x.sem()),  # Standard error of the mean
                    ('count', 'count')  # Number of observations for reference
                ]).reset_index()
                
                # Rename the columns for clarity
                summary = summary.rename(columns={'mean': f'{metric}_mean', 'sem': f'{metric}_sem', 'count': f'{metric}_count'})
                
                # Add to our list of metrics
                condition_metrics.append(summary)
        
        # Merge all metrics into a single dataframe
        if condition_metrics:
            # Start with the first metric
            condition_summary = condition_metrics[0]
            
            # Merge with remaining metrics
            for i in range(1, len(condition_metrics)):
                condition_summary = condition_summary.merge(
                    condition_metrics[i],
                    on=['experimental_condition', 'domain'],
                    how='outer'
                )
            
            print(f"Created condition summary with means and standard errors. Shape: {condition_summary.shape}")
            
            # Save the summary data
            condition_summary.to_excel('combined_analysis/condition_summary_with_sem.xlsx', index=False)
            
            # Plot grouped bar charts with error bars for each metric
            for metric in ['attention', 'usability', 'understanding', 'total_correct', 'total_rounds', 'accuracy_ratio']:
                mean_col = f'{metric}_mean'
                sem_col = f'{metric}_sem'
                
                if mean_col in condition_summary.columns and sem_col in condition_summary.columns:
                    plt.figure(figsize=(12, 7))
                    
                    # Create a bar plot with error bars
                    ax = sns.barplot(
                        data=condition_summary,
                        x='domain',
                        y=mean_col,
                        hue='experimental_condition',
                        capsize=0.1,  # Width of error bar caps
                        errwidth=1.5,  # Thickness of error bars
                        errcolor='black',  # Color of error bars
                        alpha=0.8  # Transparency of bars
                    )
                    
                    # Manually add error bars for more control
                    # Group the bars
                    conditions = condition_summary['experimental_condition'].unique()
                    domains = condition_summary['domain'].unique()
                    n_domains = len(domains)
                    n_conditions = len(conditions)
                    width = 0.8 / n_conditions  # Bar width
                    
                    # Add individual error bars with specific styling
                    for i, condition in enumerate(conditions):
                        condition_data = condition_summary[condition_summary['experimental_condition'] == condition]
                        
                        # Calculate bar positions
                        positions = np.arange(n_domains) + (i - n_conditions/2 + 0.5) * width
                        
                        # Add error bars
                        plt.errorbar(
                            x=positions,
                            y=condition_data[mean_col],
                            yerr=condition_data[sem_col],
                            fmt='none',  # No line connecting points
                            ecolor='black',  # Error bar color
                            elinewidth=1.5,  # Error bar line width
                            capsize=5,  # Cap size
                            capthick=1.5  # Cap thickness
                        )
                    
                    # Customize the plot appearance
                    plt.title(f'{metric.capitalize()} by Domain and Experimental Condition (with SEM)', fontsize=14)
                    plt.xlabel('Domain', fontsize=12)
                    plt.ylabel(f'{metric.capitalize()}', fontsize=12)
                    
                    # Add sample size information in the title or as text
                    plt.figtext(
                        0.5, 0.01, 
                        f"Error bars represent standard error of the mean (SEM)", 
                        ha='center', fontsize=10
                    )
                    
                    plt.tight_layout()
                    plt.savefig(f'combined_analysis/{metric}_by_condition_with_sem.png', dpi=300)
                    plt.close()
                    
                    # Also create a version with individual data points overlaid
                    plt.figure(figsize=(12, 7))
                    
                    # First create the bar plot
                    ax = sns.barplot(
                        data=condition_summary,
                        x='domain',
                        y=mean_col,
                        hue='experimental_condition',
                        alpha=0.7,
                        capsize=0.1
                    )
                    
                    # Then overlay the individual data points
                    sns.stripplot(
                        data=combined_df,
                        x='domain',
                        y=metric,
                        hue='experimental_condition',
                        dodge=True,
                        jitter=True,
                        marker='o',
                        alpha=0.5,
                        size=4,
                        palette='dark'
                    )
                    
                    # Handle duplicate legend
                    handles, labels = ax.get_legend_handles_labels()
                    plt.legend(handles[:n_conditions], labels[:n_conditions], title='Experimental Condition')
                    
                    plt.title(f'{metric.capitalize()} by Domain and Condition with Individual Data Points', fontsize=14)
                    plt.tight_layout()
                    plt.savefig(f'combined_analysis/{metric}_by_condition_with_points.png', dpi=300)
                    plt.close()
    
    ############

    # Add this code after the bar chart visualizations

    print("\n----- Running Two-way ANOVA Tests -----")

    # Import necessary statistical libraries
    from scipy import stats
    import statsmodels.api as sm
    from statsmodels.formula.api import ols
    import matplotlib.pyplot as plt
    import seaborn as sns
    import pandas as pd
    import numpy as np

    # Create a dataframe to store all ANOVA results
    anova_results = []

    # Define metrics to test
    metrics_to_test = ['attention', 'usability', 'understanding', 'total_correct', 'total_rounds', 
                    'accuracy_ratio', 'correct_per_round']

    # First, check which metrics are actually in the dataframe
    available_metrics = [m for m in metrics_to_test if m in combined_df.columns]
    print(f"Running two-way ANOVA tests for metrics: {available_metrics}")

    # Create directory for results if it doesn't exist
    import os
    os.makedirs('combined_analysis/two_way_anova', exist_ok=True)

    # Run two-way ANOVA for each metric
    for metric in available_metrics:
        try:
            print(f"\nAnalyzing metric: {metric}")
            
            # Create formula for the model: main effects and interaction
            formula = f"{metric} ~ C(experimental_condition) + C(domain) + C(experimental_condition):C(domain)"
            
            # Fit the model
            model = ols(formula, data=combined_df).fit()
            
            # Run ANOVA Type II (accounts for unbalanced designs)
            anova_table = sm.stats.anova_lm(model, typ=2)
            
            # Print the ANOVA table
            print(f"Two-way ANOVA results for {metric}:")
            print(anova_table)
            
            # Extract results for the three effects (experimental_condition, domain, interaction)
            effects = ['C(experimental_condition)', 'C(domain)', 'C(experimental_condition):C(domain)']
            effect_names = ['Experimental Condition', 'Domain', 'Interaction']
            
            for effect, effect_name in zip(effects, effect_names):
                if effect in anova_table.index:
                    # Extract statistics
                    f_stat = anova_table.loc[effect, 'F']
                    p_val = anova_table.loc[effect, 'PR(>F)']
                    df1 = anova_table.loc[effect, 'df']
                    df2 = anova_table.loc['Residual', 'df']
                    
                    # Determine significance level
                    significance = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
                    
                    # Store results
                    anova_results.append({
                        'Metric': metric,
                        'Effect': effect_name,
                        'F_statistic': f_stat,
                        'p_value': p_val,
                        'Significance': significance,
                        'df_effect': df1,
                        'df_residual': df2,
                        'Sum_Sq_effect': anova_table.loc[effect, 'sum_sq'],
                        'Sum_Sq_residual': anova_table.loc['Residual', 'sum_sq']
                    })
                    
                    # Print result in APA format
                    print(f"  {effect_name}: F({df1:.0f},{df2:.0f})={f_stat:.2f}, p={p_val:.4f} {significance}")
            
            # If there's a significant interaction, create an interaction plot
            if 'C(experimental_condition):C(domain)' in anova_table.index:
                p_val = anova_table.loc['C(experimental_condition):C(domain)', 'PR(>F)']
                
                if p_val < 0.05:
                    print("  Significant interaction detected. Creating interaction plot...")
                    
                    # Calculate means for each combination of domain and condition
                    interaction_means = combined_df.groupby(['domain', 'experimental_condition'])[metric].mean().reset_index()
                    
                    # Create the interaction plot
                    plt.figure(figsize=(12, 8))
                    
                    # Plot lines for each domain
                    interaction_pivot = interaction_means.pivot(index='domain', columns='experimental_condition', values=metric)
                    interaction_pivot.plot(marker='o', linewidth=2)
                    
                    plt.title(f'Interaction Plot: {metric.capitalize()} by Domain and Experimental Condition', fontsize=14)
                    plt.xlabel('Domain', fontsize=12)
                    plt.ylabel(metric.capitalize(), fontsize=12)
                    plt.grid(True, linestyle='--', alpha=0.7)
                    plt.legend(title='Experimental Condition')
                    plt.tight_layout()
                    plt.savefig(f'combined_analysis/two_way_anova/{metric}_interaction_plot.png', dpi=300)
                    plt.close()
                    
                    # Also create a grouped bar chart for the interaction
                    plt.figure(figsize=(12, 8))
                    sns.barplot(
                        data=interaction_means,
                        x='domain',
                        y=metric,
                        hue='experimental_condition',
                        capsize=0.1
                    )
                    plt.title(f'Interaction: {metric.capitalize()} by Domain and Experimental Condition', fontsize=14)
                    plt.xlabel('Domain', fontsize=12)
                    plt.ylabel(metric.capitalize(), fontsize=12)
                    plt.tight_layout()
                    plt.savefig(f'combined_analysis/two_way_anova/{metric}_interaction_bars.png', dpi=300)
                    plt.close()
            
            # If there's a significant main effect of experimental condition, run post-hoc tests
            if 'C(experimental_condition)' in anova_table.index:
                p_val = anova_table.loc['C(experimental_condition)', 'PR(>F)']
                
                if p_val < 0.05:
                    print("  Significant main effect of experimental condition. Running post-hoc tests...")
                    
                    # Get experimental conditions
                    conditions = combined_df['experimental_condition'].unique()
                    
                    # Initialize post-hoc results
                    posthoc_results = []
                    
                    # For each pair of conditions, run t-test
                    for i in range(len(conditions)):
                        for j in range(i+1, len(conditions)):
                            cond1 = conditions[i]
                            cond2 = conditions[j]
                            
                            cond1_data = combined_df[combined_df['experimental_condition'] == cond1][metric]
                            cond2_data = combined_df[combined_df['experimental_condition'] == cond2][metric]
                            
                            if len(cond1_data) > 0 and len(cond2_data) > 0:
                                t_stat, p_value = stats.ttest_ind(cond1_data, cond2_data, equal_var=False)
                                sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
                                
                                posthoc_results.append({
                                    'Metric': metric,
                                    'Comparison': f"{cond1} vs {cond2}",
                                    'Condition1': cond1,
                                    'Condition2': cond2,
                                    't_statistic': t_stat,
                                    'p_value': p_value,
                                    'Significance': sig,
                                    'Mean1': cond1_data.mean(),
                                    'Mean2': cond2_data.mean(),
                                    'Difference': cond1_data.mean() - cond2_data.mean()
                                })
                                
                                print(f"    {cond1} vs {cond2}: t={t_stat:.2f}, p={p_value:.4f} {sig}")
                    
                    # Save post-hoc results
                    if posthoc_results:
                        posthoc_df = pd.DataFrame(posthoc_results)
                        posthoc_df.to_excel(f'combined_analysis/two_way_anova/{metric}_posthoc_tests.xlsx', index=False)
        
        except Exception as e:
            print(f"Error running two-way ANOVA for {metric}: {e}")
            import traceback
            traceback.print_exc()

    # Convert results to dataframe
    anova_df = pd.DataFrame(anova_results)

    # Save results to Excel
    if not anova_df.empty:
        anova_df.to_excel('combined_analysis/two_way_anova_results.xlsx', index=False)
        print(f"\nSaved two-way ANOVA results to 'combined_analysis/two_way_anova_results.xlsx'")
        
        # Create summary visualizations
        
        # 1. Heatmap of p-values for all effects
        plt.figure(figsize=(12, 8))
        
        # Create pivot table
        pivot_data = anova_df.pivot(index='Metric', columns='Effect', values='p_value')
        
        # Create the heatmap
        ax = sns.heatmap(
            pivot_data,
            annot=True,
            fmt='.4f',
            cmap='RdYlGn_r',  # Red-Yellow-Green reversed
            vmin=0,
            vmax=0.1,
            linewidths=0.5,
            cbar_kws={'label': 'p-value'}
        )
        
        # Add significance markers
        sig_pivot = anova_df.pivot(index='Metric', columns='Effect', values='Significance')
        
        # Loop through cells to add significance markers
        for i in range(pivot_data.shape[0]):
            for j in range(pivot_data.shape[1]):
                try:
                    sig = sig_pivot.iloc[i, j]
                    if pd.notna(sig) and sig != 'ns':
                        ax.text(j + 0.5, i + 0.85, sig, 
                            horizontalalignment='center', verticalalignment='center',
                            color='black', fontweight='bold')
                except:
                    pass
        
        plt.title('Two-way ANOVA p-values by Effect and Metric', fontsize=14)
        plt.tight_layout()
        plt.savefig('combined_analysis/two_way_anova_pvalues.png', dpi=300)
        plt.close()
        
        # 2. Bar chart of F-statistics for significant results
        significant_results = anova_df[anova_df['p_value'] < 0.05].copy()
        
        if len(significant_results) > 0:
            plt.figure(figsize=(14, 8))
            
            # Create a grouped bar chart
            sns.barplot(
                data=significant_results,
                x='Metric',
                y='F_statistic',
                hue='Effect'
            )
            
            # Add p-value annotations
            for i, row in enumerate(significant_results.itertuples()):
                plt.text(
                    i % len(available_metrics), row.F_statistic + 0.5, 
                    f"p={row.p_value:.4f} {row.Significance}",
                    ha='center', va='bottom',
                    rotation=90, fontsize=8
                )
            
            plt.title('Significant Two-way ANOVA Results (p < 0.05)', fontsize=14)
            plt.xlabel('Metric', fontsize=12)
            plt.ylabel('F-statistic', fontsize=12)
            plt.xticks(rotation=45)
            plt.legend(title='Effect Type')
            plt.tight_layout()
            plt.savefig('combined_analysis/significant_two_way_anova.png', dpi=300)
            plt.close()

        # 3. Create effect size plot
        if len(anova_df) > 0:
            # Calculate eta-squared (effect size)
            for i, row in anova_df.iterrows():
                total_ss = row['Sum_Sq_effect'] + row['Sum_Sq_residual']
                anova_df.loc[i, 'eta_squared'] = row['Sum_Sq_effect'] / total_ss if total_ss > 0 else 0
            
            # Create effect size visualization
            plt.figure(figsize=(14, 8))
            
            # Create heatmap of effect sizes
            effect_pivot = anova_df.pivot(index='Metric', columns='Effect', values='eta_squared')
            
            sns.heatmap(
                effect_pivot,
                annot=True,
                fmt='.3f',
                cmap='YlGnBu',
                vmin=0,
                vmax=0.5,  # Adjust max value based on your data
                linewidths=0.5,
                cbar_kws={'label': 'Eta-squared (effect size)'}
            )
            
            plt.title('Effect Sizes (Eta-squared) for Two-way ANOVA', fontsize=14)
            plt.tight_layout()
            plt.savefig('combined_analysis/two_way_anova_effect_sizes.png', dpi=300)
            plt.close()
    else:
        print("No two-way ANOVA results to save.")
    ############
    # 4. Scatter plot matrix of key variables
    if len(combined_df) > 0:
        key_vars = []
        for var in ['attention', 'understanding', 'total_rounds', 'total_correct', 'accuracy_ratio']:
            if var in combined_df.columns:
                key_vars.append(var)
        
        if len(key_vars) >= 2:
            plt.figure(figsize=(15, 15))
            sns.pairplot(
                combined_df,
                vars=key_vars,
                hue='domain',
                diag_kind='kde',
                plot_kws={'alpha': 0.6}
            )
            plt.tight_layout()
            plt.savefig('combined_analysis/metrics_pairplot.png', dpi=300)
            plt.close()
    
    print("Visualizations of combined analysis saved to 'combined_analysis' directory")
except Exception as e:
    print(f"Error during visualization of combined data: {e}")


In [ ]:
def plot_prob_learning():
    # # Load the data from the Excel file
    # file_path = 'output.xlsx'
    # round_data = pd.read_excel(file_path, sheet_name='round')
    # domain_params_data = pd.read_excel(file_path, sheet_name='domain_params')

    with open('round.pkl', 'rb') as f:
        round_data = pickle.load(f)

    with open('domain_params.pkl', 'rb') as f:
        domain_params_data = pickle.load(f)

    # # Extract 'min_BC_constraints' from the domain_params sheet
    # domain_params_data['min_BC_constraints'] = domain_params_data['min_BEC_constraints'].apply(
    #     lambda x: np.array(eval(x)[0]) if isinstance(x, str) else None
    # )


    # Group data by domain, group, and player for separate analysis
    grouped_round_data = round_data.groupby(['domain', 'group_id'])

    # Process each group for separate plotting
    for (domain, group), group_data in grouped_round_data:
        print(f"\nDomain: {domain}, Group: {group}")
        if group==18:
            rounds = group_data['round_num'].tolist()
            statuses = group_data['status'].tolist()
            kc_ids = group_data['kc_id'].tolist()

            prob_unit_learning = {}
            prob_BEC_learning = {}
            prob_learning = {}
            member_info = {}

            for rnd_id, rnd in group_data.iterrows():
                members_statuses = rnd.members_statuses
                min_kc_constraints = np.array(rnd['min_KC_constraints'])
                min_bec_constraints = np.array(rnd['min_BEC_constraints_running'])
                domain_constraints = np.array(domain_params_data[(domain_params_data['domain_name'] == domain)]['min_BEC_constraints'])

                if domain == 'sb':
                    domain_constraints = np.array([np.array([[5, 2, 5]]), np.array([[ 3, -3,  1]]), np.array([[-6,  4, -3]])])
                elif domain == 'at':
                    domain_constraints = np.array([np.array([[1, 1, 0]]), np.array([[ 0, -1, -4]]), np.array([[-1,  0,  2]])])

                # print('Domain constraints:', domain_constraints)

                
                for model_id in range(3):    
                    if members_statuses[model_id] == 'joined':   
                        particles_pos = np.array(rnd.ind_member_models_pos[0][model_id])
                        particles_weights = np.array(rnd.ind_member_models_weights[0][model_id])

                        # print(particles_pos)
                        # print(particles_weights)
                        # print(min_kc_constraints)
                        # print(min_bec_constraints)
                        # print(domain_constraints)

                        member_str = f'Rnd: {rnd_id}, Model: {model_id}'

                        pf = pf_team.Particles_team(particles_pos, 0.8, 36)
                        pf.weights = particles_weights

                        member_info[model_id] = member_str

                        # Calculate probabilities
                        if model_id not in prob_unit_learning:
                            prob_unit_learning[model_id] = []
                            prob_BEC_learning[model_id] = []
                            prob_learning[model_id] = []
                        pf.calc_particles_probability(min_kc_constraints)
                        prob_unit_learning[model_id].append(pf.particles_prob_correct)
                        # print('prob_unit_learning:', prob_unit_learning)

                        pf.calc_particles_probability(min_bec_constraints)
                        prob_BEC_learning[model_id].append(pf.particles_prob_correct)
                        # print('prob_BEC_learning:', prob_BEC_learning)

                        pf.calc_particles_probability(domain_constraints)
                        prob_learning[model_id].append(pf.particles_prob_correct)
                        # print('prob_learning:', prob_learning)
                    


    # Plot the probabilities for the current domain, group, and player
    for model_id in range(3):
        if len(prob_unit_learning[model_id]) > 0:  # Only plot if data exists
            plt.figure(figsize=(12, 6))
            plt.plot(rounds, prob_unit_learning[model_id], label=f'Model {model_id} Probability Unit Learning', marker='o', linestyle='-')
            plt.plot(rounds, prob_BEC_learning[model_id], label=f'Model {model_id} Probability BEC Learning', marker='x', linestyle='--')
            plt.plot(rounds, prob_learning[model_id], label=f'Model {model_id} Probability BC Learning', marker='s', linestyle=':')
            plt.xlabel('Round')
            plt.ylabel('Probability of Learning')
            plt.title(f'Learning Probabilities for Domain: {domain}, Group: {group}, Player: {model_id}')
            plt.legend()
            plt.grid()
            plt.show()

#####################################

# plot_prob_learning()
